# 서번트 텍스트 생성 파이프라인

**초기 파라미터**: 이름, 성별, 클래스, 속성을 입력받아 진행합니다.

# 1. 인물의 세계관 설정 분석

In [ ]:
from pydantic import BaseModel
from typing import Literal, Optional


class LoreMappingResult(BaseModel):
    name: str
    gender: Literal["Male", "Female"]

    # 세계관 분석 결과
    historical_or_mythical: Literal[
        "Historical",   # 역사적
        "Mythical",    # 신화적
        "Legendary",  # 전설적
        "Conceptual"  # 개념적
    ]
    origin_country: Optional[str]
    era: str

    # Fate식 해석
    main_archetype: Literal[
        "Divine King",  # 신화왕
        "Tyrant King",  # 타락왕
        "Conqueror",    # 정복자
        "Saint",       # 성인
        "Naval Commander",  # 해군 지휘관
        "Trickster",     # 꼼수꾼
        "Magus King",    # 마술왕
        "Heroic Spirit" # 영웅 정신
    ]

    likely_class_candidates: list[str]

    # 전설성 / 신비도
    legend_rank: Literal[
        "Low",        # 저
        "Medium",     # 중
        "High",       # 고
        "Extreme"     # 극
    ]

    # 신비도
    mystery_level: Literal[
        "Modern",     # 현대
        "Medieval",   # 중세
        "Ancient",    # 고대
        "Age of Gods" # 신화시대
    ]

    # 신성 후보
    divinity_potential: Literal[
        "None",       # 없음
        "Low",        # 낮음
        "Medium",     # 중간
        "High"        # 높음
    ]

    # 핵심 상징
    iconic_weapons_or_symbols: list[str]
    key_achievements: list[str]

    # strange Fake용 플래그
    suitable_for_pretender: bool
    suitable_for_foreigner: bool


In [2]:
from openai import OpenAI
from pydantic import ValidationError
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

def run_lore_mapping(name: str, gender: str) -> LoreMappingResult:
    system_prompt = """
You are a Fate/strange Fake and Fate/stay night lore analysis engine.
Analyze the given character according to Nasuverse rules.
Return ONLY valid JSON matching the provided schema.
Do NOT include extra commentary.
Only mark suitable_for_pretender as true if the character is known
for identity fraud, impersonation, false legends, or multiple historical identities.
"""

    user_prompt = f"""
Name: {name}
Gender: {gender}

Analyze this character for Fate/strange Fake universe adaptation.
"""

    # Responses API: 인자는 input, text_format (Chat Completions의 messages, response_format 아님)
    response = client.responses.parse(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=LoreMappingResult,
    )

    try:
        result: LoreMappingResult = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


# 2. 최적의 설정 탐색

In [ ]:
from pydantic import BaseModel
from typing import Literal


class OptimalServantSetup(BaseModel):
    # 핵심 Fate 스펙
    class_name: Literal[
        "Saber",    # 세이버
        "Archer",  # 아처
        "Lancer",  # 랜서
        "Rider",   # 라이더
        "Caster",  # 캐스터
        "Assassin", # 어새신   
        "Berserker", # 버서커
        "Ruler",    # 룰러
        "Avenger",  # 어벤저
        "Alter Ego", # 얼터 에고
        "Pretender", # 프리텐더
        "Foreigner" # 포리너
    ]
    spirit_origin_type: str
    attribute: Literal[
        "Heaven",   # 천
        "Earth",    # 지
        "Human",    # 인
        "Star",     # 별
        "Beast"     # 짐승
    ]
    alignment: str
    divinity_rank: str
    era: str

    # 전투/보구
    main_noble_phantasm_type: str

    # 
    world_threat_level: Literal[
        "Local",        # 지역
        "National",    # 국가
        "Continental",  # 대륙
        "Mythic",      # 신화
        "World-Class" # 세계적
    ]

    # 설정 요약
    core_concept: str
    representative_noble_phantasm: str


In [4]:
from openai import OpenAI
from pydantic import ValidationError

client = OpenAI()

def run_optimal_servant_setup(
    lore: LoreMappingResult,
    initial_class: str | None = None,
    initial_attribute: str | None = None,
) -> OptimalServantSetup:
    system_prompt = """
You are a Fate/strange Fake character designer.
Using the provided lore mapping, generate ONE optimal Servant setup.

STRICT RULES:
- Do NOT generate any image prompts.
- Do NOT include character appearance or moe design.
- Focus ONLY on Fate/Nasuverse combat and lore configuration.
- If initial class and/or attribute are given, you MUST use them for class_name and attribute.

Return ONLY valid JSON matching the schema.
"""

    user_prompt = f"""
Lore Mapping Result (JSON):

{lore.model_dump_json(indent=2)}
"""
    if initial_class:
        user_prompt += f"\nUse this Servant class: {initial_class}.\n"
    if initial_attribute:
        user_prompt += f"Use this attribute: {initial_attribute}.\n"
    user_prompt += "\nGenerate the optimal single Servant configuration.\n"

    response = client.responses.parse(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=OptimalServantSetup,
    )

    try:
        result: OptimalServantSetup = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


# 3. 이미지 생성 프롬프트 출력력

In [5]:
from pydantic import BaseModel


class ImagePromptResult(BaseModel):
    # 최종 이미지 생성용 (영문)
    landscape_image_prompt_en: str

    # 네거티브/제거 옵션 (선택)
    negative_prompt_en: str


In [6]:
from openai import OpenAI
from pydantic import ValidationError

client = OpenAI()

def run_image_prompt_generator(
    lore: LoreMappingResult,
    servant: OptimalServantSetup,
) -> ImagePromptResult:
    system_prompt = """
You are an anime illustration prompt engineer for Fate-style characters.

TASK:
- Generate ONLY image generation prompts.
- Use moe-style character design.
- Respect the Servant class, era, and core concept.
- Apply natural Fate-style gender adaptation.
- Create a wide horizontal (16:9) cinematic composition.

STRICT RULES:
- Do NOT change lore or Servant settings.
- Do NOT add new Noble Phantasms.
- Do NOT include gameplay or combat stats.
- NO text, NO UI, NO watermark, NO logo, NO letters in image.

Return ONLY valid JSON matching the schema.
"""

    user_prompt = f"""
Lore Mapping (JSON):
{lore.model_dump_json(indent=2)}

Optimal Servant Setup (JSON):
{servant.model_dump_json(indent=2)}

Generate a Fate-style anime illustration prompt
for this Servant in landscape (16:9) format.
"""

    response = client.responses.parse(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=ImagePromptResult,
    )

    try:
        result: ImagePromptResult = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


In [7]:
def normalize_image_prompt(prompt: str) -> str:
    required_tokens = [
        "anime illustration",
        "Type-Moon official art style",
        "cinematic lighting",
        "dramatic background",
        "wide horizontal composition",
        "16:9 landscape",
        "no text",
        "no UI",
        "no watermark",
        "no logo",
        "no letters",
    ]

    normalized = prompt.strip()

    lower = normalized.lower()
    for token in required_tokens:
        if token.lower() not in lower:
            normalized += f", {token}"

    return normalized


In [8]:
def normalize_negative_prompt(negative: str) -> str:
    required_negative = [
        "text",
        "logo",
        "watermark",
        "UI",
        "signature",
        "subtitles",
        "low resolution",
        "blurry",
        "bad anatomy",
        "extra fingers",
        "extra arms",
        "deformed",
    ]

    normalized = negative.strip()
    lower = normalized.lower()

    for token in required_negative:
        if token.lower() not in lower:
            normalized += f", {token}"

    return normalized


In [9]:
def run_prompt_postprocess(step3: ImagePromptResult) -> ImagePromptResult:
    cleaned_prompt = normalize_image_prompt(step3.landscape_image_prompt_en)
    cleaned_negative = normalize_negative_prompt(step3.negative_prompt_en)

    return ImagePromptResult(
        landscape_image_prompt_en=cleaned_prompt,
        negative_prompt_en=cleaned_negative,
    )


# 4. 카드 데이터로 출력

In [ ]:
from pydantic import BaseModel
from typing import Literal


ElementTypeKR = Literal["불", "물", "땅", "바람", "빛", "어둠"]
ServantClassType = Literal[
    "Saber",    # 세이버
    "Archer",   # 아처
    "Lancer",   # 랜서
    "Rider",    # 라이더
    "Caster",   # 캐스터
    "Assassin", # 어새신
    "Berserker", # 버서커
    "Ruler",    # 룰러
    "Avenger",  # 어벤저
]


class ServantCardStats(BaseModel):
    # 카드 기본 정보 (한글)
    card_name: str                 # 카드명 (한글)
    card_type: ServantClassType   # 타입 = 클래스
    element: ElementTypeKR        # 속성 (불/물/땅/바람/빛/어둠)
    rarity: Literal[1, 2, 3, 4, 5] # 등급

    # 전투 스탯
    attack: int
    health: int

    # 보구 - 한글
    skill_name_1: str
    skill_desc_1: str   # 한줄

    skill_name_2: str
    skill_desc_2: str   # 한줄

    # 플레이버 텍스트 - 한줄
    flavor_text: str


In [ ]:
from openai import OpenAI
from pydantic import ValidationError

client = OpenAI()

def run_card_stat_generator(
    lore: LoreMappingResult,
    servant: OptimalServantSetup,
) -> ServantCardStats:
    system_prompt = """
당신은 Fate 스타일 서번트 카드 게임 디자이너입니다.

목표:
- 카드 결과물은 반드시 한글로 작성하십시오.
- 카드 타입은 서번트 클래스입니다.
- 속성은 반드시 다음 중 하나입니다: 불, 물, 땅, 바람, 빛, 어둠
- 스킬(보구)은 반드시 보구를 기반으로 합니다.
- 각 스킬 설명과 플레이버 텍스트는 반드시 한줄로 작성하십시오.
- 각 스킬 설명과 플레이버 텍스트(대사)는 각각 25자를 초과하지 않도록 작성하십시오.

규칙:
- card_type은 Servant 클래스와 반드시 일치해야 합니다.
- 카드명, 스킬명(보구명), 스킬설명(보구진명개방), 플레이버 텍스트는 전부 한글이어야 합니다.
- 이미지 프롬프트나 추가 설정은 생성하지 마십시오.
- 게임 밸런스를 고려하여 등급에 맞는 수치를 설정하십시오.

밸런스 가이드:
- 1~2성: 낮음
- 3성: 중간
- 4성: 높음
- 5성: 최상급

반드시 스키마에 맞는 JSON만 출력하십시오.
"""

    user_prompt = f"""
기본 서번트 정보 (JSON):

{lore.model_dump_json(indent=2)}

최적 서번트 설정 (JSON):

{servant.model_dump_json(indent=2)}

위 설정을 기반으로 한글 서번트 카드 객체를 생성하십시오.
"""

    response = client.responses.parse(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=ServantCardStats,
    )

    try:
        result: ServantCardStats = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


In [12]:
# ========== 초기 파라미터 (이름 / 성별 / 클래스 / 속성) ==========
# 아래 값을 수정하여 실행하세요.

INITIAL_NAME = "스파르타쿠스"           # 이름 (한글/영문)
INITIAL_GENDER = "남성"             # 성별: "남성" / "여성"
INITIAL_CLASS = "버서커"             # 클래스: Saber, Archer, Lancer, Rider, Caster, Assassin, Berserker, Ruler, Avenger, Alter Ego, Pretender, Foreigner
INITIAL_ATTRIBUTE = "Human"        # 속성: Heaven, Earth, Human, Star, Beast

print(f"입력: 이름={INITIAL_NAME}, 성별={INITIAL_GENDER}, 클래스={INITIAL_CLASS}, 속성={INITIAL_ATTRIBUTE}")

입력: 이름=스파르타쿠스, 성별=남성, 클래스=버서커, 속성=Human


In [13]:
# Step 1 (초기 파라미터: 이름, 성별 사용)
step1_result = run_lore_mapping(
    name=INITIAL_NAME,
    gender=INITIAL_GENDER,
)

print("=== Step 1: Lore Mapping ===")
print(step1_result.model_dump_json(indent=2))


# Step 2 (초기 파라미터: 클래스, 속성 사용)
step2_result = run_optimal_servant_setup(
    step1_result,
    initial_class=INITIAL_CLASS,
    initial_attribute=INITIAL_ATTRIBUTE,
)

print("\n=== Step 2: Optimal Servant Setup ===")
print(step2_result.model_dump_json(indent=2))


# Step 3
step3_result = run_image_prompt_generator(step1_result, step2_result)

print("\n=== Step 3: Image Prompt ===")
print(step3_result.model_dump_json(indent=2))

# 3.5
step3_5 = run_prompt_postprocess(step3_result)

print('# Prompt')
print(' ' + step3_5.landscape_image_prompt_en)
print('# Negative Prompt')
print(' ' + step3_5.negative_prompt_en)

# Step 4
step4_result = run_card_stat_generator(step1_result, step2_result)

print("\n=== Step 4: Servant Card Stats (KR) ===")
print(step4_result.model_dump_json(indent=2))


=== Step 1: Lore Mapping ===
{
  "name": "스파르타쿠스",
  "gender": "Male",
  "historical_or_mythical": "Historical",
  "origin_country": "Roman Empire",
  "era": "Ancient",
  "main_archetype": "Heroic Spirit",
  "likely_class_candidates": [
    "Berserker",
    "Ruler"
  ],
  "legend_rank": "High",
  "mystery_level": "Ancient",
  "divinity_potential": "None",
  "iconic_weapons_or_symbols": [
    "Gladius",
    "Roman standards"
  ],
  "key_achievements": [
    "Led a major slave uprising against the Roman Republic",
    "Defeated Roman legions in battle"
  ],
  "suitable_for_pretender": false,
  "suitable_for_foreigner": false
}

=== Step 2: Optimal Servant Setup ===
{
  "class_name": "Berserker",
  "spirit_origin_type": "Heroic Spirit",
  "attribute": "Human",
  "alignment": "Neutral Good",
  "divinity_rank": "None",
  "era": "Ancient",
  "main_noble_phantasm_type": "Anti-Army",
  "world_threat_level": "National",
  "core_concept": "Rebellion against oppression",
  "representative_noble_p

In [1]:
step4_result

NameError: name 'step4_result' is not defined